# Figure 4: Hard Generalization On Norman

- Figure / panels: `Fig4a`, `Fig4b`, `Fig4d` are main-text defaults; `Fig4c` is optional supplementary reserve
- Inputs: `artifacts/results/norman/metrics_nearest.csv`, `artifacts/results/<baseline>/norman/metrics.csv`, Norman payload `pkl`
- Outputs: `artifacts/paper_figures/main/Fig4_NormanGeneralization/*`
- Run order: run after Fig2; this notebook is the Norman subgroup and unseen-combo module
- Dataset selector: edit `DATASET` in the parameter cell below; this notebook currently supports `norman` only
- Role: Main-text stress-test module; `Fig4c` is off by default


In [ ]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'scripts').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

sns.set_theme(style='ticks', font_scale=1.0)
plt.rcParams['figure.dpi'] = 220
plt.rcParams['savefig.dpi'] = 220

from scripts.trishift.analysis import baseline_panel
from scripts.trishift.analysis._result_adapter import load_metrics_df, load_payload_item, resolve_result
from trishift._utils import normalize_condition
importlib.reload(baseline_panel)


In [ ]:
AVAILABLE_DATASETS = ['norman']
DATASET = 'norman'  # edit here
if DATASET not in AVAILABLE_DATASETS:
    raise ValueError(f'Unsupported dataset: {DATASET}. Available: {AVAILABLE_DATASETS}')

MODELS = ['trishift_nearest', 'biolord', 'gears', 'genepert', 'scgpt']  # edit here
AVAILABLE_SPLIT_IDS = [1, 2, 3, 4, 5]
SELECTED_SPLIT_IDS = AVAILABLE_SPLIT_IDS.copy()  # edit here
EXPORT_FIG4C = False  # edit here; set True only when the hard-regime panel is favorable enough to keep
invalid_split_ids = [split_id for split_id in SELECTED_SPLIT_IDS if split_id not in AVAILABLE_SPLIT_IDS]
if invalid_split_ids:
    raise ValueError(f'Unknown split ids: {invalid_split_ids}. Available: {AVAILABLE_SPLIT_IDS}')
SPLIT_IDS = [int(split_id) for split_id in SELECTED_SPLIT_IDS]
OUT_ROOT = repo_root / 'artifacts' / 'paper_figures' / 'main' / 'Fig4_NormanGeneralization'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_LABELS = {'trishift_nearest': 'TriShift nearest', 'biolord': 'biolord', 'gears': 'GEARS', 'genepert': 'GenePert', 'scgpt': 'scGPT', 'Truth': 'Truth'}
MODEL_COLORS = {'TriShift nearest': '#4C78A8', 'biolord': '#0073C2', 'GEARS': '#D04E59', 'GenePert': '#54A24B', 'scGPT': '#B279A2', 'Truth': '#7F7F7F'}
print('Selected dataset:', DATASET)
print('Selected splits:', SPLIT_IDS)
print('Selected models:', MODELS)
print('Export Fig4c:', EXPORT_FIG4C)


In [ ]:
result = baseline_panel.run_baseline_panel(dataset=DATASET, models=MODELS, split_ids=SPLIT_IDS, out_root=OUT_ROOT / 'benchmark')
summary_df = result['summary_df'].copy()
subgroup_df = result['subgroup_df'].copy()
summary_df.to_csv(OUT_ROOT / 'fig4_summary.csv', index=False, encoding='utf-8-sig')
subgroup_df.to_csv(OUT_ROOT / 'fig4_subgroup_summary.csv', index=False, encoding='utf-8-sig')
display(summary_df)
display(subgroup_df.head())


In [ ]:
metrics_path = repo_root / 'artifacts' / 'results' / 'norman' / 'metrics_nearest.csv'
if metrics_path.exists():
    metrics_df = pd.read_csv(metrics_path)
    schema_df = metrics_df[['condition', 'subgroup']].drop_duplicates().groupby('subgroup', as_index=False).size().rename(columns={'size': 'n_conditions'})
else:
    schema_df = pd.DataFrame(columns=['subgroup', 'n_conditions'])

schema_df.to_csv(OUT_ROOT / 'fig4a_subgroup_schema.csv', index=False, encoding='utf-8-sig')
fig, ax = plt.subplots(figsize=(6.3, 4.0), dpi=220)
if schema_df.empty:
    ax.text(0.5, 0.5, 'Norman subgroup counts unavailable', ha='center', va='center'); ax.axis('off')
else:
    sns.barplot(data=schema_df, x='subgroup', y='n_conditions', order=['single', 'seen2', 'seen1', 'seen0'], color='#4C78A8', ax=ax)
    ax.set_xlabel(''); ax.set_ylabel('Number of conditions'); ax.set_title('Fig4a: Norman subgroup split')
    for patch in ax.patches:
        height = patch.get_height(); ax.text(patch.get_x() + patch.get_width() / 2, height, f'{int(height)}', ha='center', va='bottom', fontsize=9)
fig.tight_layout(); fig.savefig(OUT_ROOT / 'fig4a_subgroup_schema.png', bbox_inches='tight'); plt.close(fig)


In [ ]:
metrics_to_plot = [('pearson', 'Pearson'), ('nmse', 'NMSE'), ('deg_mean_r2', 'DEG mean R2')]
plot_df = subgroup_df[subgroup_df['subgroup'].isin(['seen2', 'seen1', 'seen0'])].copy()
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.8), dpi=220)
for ax, (metric, metric_label) in zip(axes, metrics_to_plot):
    if plot_df.empty or f'mean_{metric}' not in plot_df.columns:
        ax.text(0.5, 0.5, f'No rows for {metric}', ha='center', va='center'); ax.axis('off'); continue
    sns.barplot(data=plot_df, x='subgroup', y=f'mean_{metric}', hue='label', order=['seen2', 'seen1', 'seen0'], palette=MODEL_COLORS, ax=ax)
    ax.set_xlabel(''); ax.set_ylabel(metric_label); ax.set_title(metric_label); ax.legend_.remove()
    for patch in ax.patches:
        patch.set_edgecolor('black'); patch.set_linewidth(0.4)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, frameon=False, title='', ncol=4, loc='upper center', bbox_to_anchor=(0.5, 1.03))
fig.suptitle('Fig4b: Norman subgroup benchmark', y=1.08)
fig.tight_layout(); fig.savefig(OUT_ROOT / 'fig4b_subgroup_benchmark.png', bbox_inches='tight'); plt.close(fig)


In [ ]:
focus_rows = []
if not plot_df.empty:
    for subgroup in ['seen1', 'seen0']:
        sub = plot_df[plot_df['subgroup'] == subgroup].copy()
        tri = sub[sub['model_name'] == 'trishift_nearest']
        competitors = sub[sub['model_name'] != 'trishift_nearest']
        if tri.empty or competitors.empty:
            continue
        focus_rows.append({'subgroup': subgroup, 'pearson_gain_vs_best': float(tri['mean_pearson'].iloc[0]) - float(competitors['mean_pearson'].max()), 'deg_mean_r2_gain_vs_best': float(tri['mean_deg_mean_r2'].iloc[0]) - float(competitors['mean_deg_mean_r2'].max())})
focus_df = pd.DataFrame(focus_rows)
if EXPORT_FIG4C:
    focus_df.to_csv(OUT_ROOT / 'fig4c_hard_regime_focus.csv', index=False, encoding='utf-8-sig')
    fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.0), dpi=220)
    if focus_df.empty:
        for ax in axes:
            ax.text(0.5, 0.5, 'No hard-regime summary available', ha='center', va='center'); ax.axis('off')
    else:
        sns.barplot(data=focus_df, x='subgroup', y='pearson_gain_vs_best', color='#4C78A8', ax=axes[0]); axes[0].set_title('Pearson gain over best baseline'); axes[0].set_xlabel('')
        sns.barplot(data=focus_df, x='subgroup', y='deg_mean_r2_gain_vs_best', color='#54A24B', ax=axes[1]); axes[1].set_title('DEG mean R2 gain over best baseline'); axes[1].set_xlabel('')
    fig.tight_layout(); fig.savefig(OUT_ROOT / 'fig4c_hard_regime_focus.png', bbox_inches='tight'); plt.close(fig)
    display(focus_df)
else:
    print('Fig4c is disabled by default. Set EXPORT_FIG4C=True only if this panel is favorable enough to keep.')


In [ ]:
per_model = []
for model_name in MODELS:
    try:
        resolved = resolve_result(dataset=DATASET, model_name=model_name)
        per_model.append(load_metrics_df(resolved).assign(model_name=model_name))
    except Exception:
        continue
all_metrics = pd.concat(per_model, ignore_index=True) if per_model else pd.DataFrame()
case_meta, case_plot_df = None, None
if not all_metrics.empty:
    avg_metrics = all_metrics.groupby(['model_name', 'condition', 'subgroup'], as_index=False)[['pearson', 'deg_mean_r2']].mean(numeric_only=True)
    tri = avg_metrics[avg_metrics['model_name'] == 'trishift_nearest'][['condition', 'subgroup', 'pearson', 'deg_mean_r2']].rename(columns={'pearson': 'pearson_trishift', 'deg_mean_r2': 'deg_trishift'})
    competitors = avg_metrics[avg_metrics['model_name'] != 'trishift_nearest'].groupby(['condition', 'subgroup'], as_index=False)[['pearson', 'deg_mean_r2']].max(numeric_only=True).rename(columns={'pearson': 'pearson_best_comp', 'deg_mean_r2': 'deg_best_comp'})
    merged = tri.merge(competitors, on=['condition', 'subgroup'], how='inner')
    merged = merged[merged['subgroup'].isin(['seen1', 'seen0'])].copy()
    if not merged.empty:
        merged['score_gain'] = (merged['pearson_trishift'] - merged['pearson_best_comp']) + (merged['deg_trishift'] - merged['deg_best_comp'])
        for best_case in merged.sort_values('score_gain', ascending=False).to_dict('records'):
            condition_key = normalize_condition(best_case['condition'])
            subgroup = str(best_case['subgroup'])
            for split_id in SPLIT_IDS:
                loaded = {}
                for model_name in MODELS:
                    try:
                        _, payload = load_payload_item(dataset=DATASET, model_name=model_name, split_id=split_id, condition=None)
                        item = {normalize_condition(k): v for k, v in payload.items()}.get(condition_key)
                        if item is not None:
                            loaded[model_name] = item
                    except Exception:
                        continue
                if len(loaded) < len(MODELS) or 'trishift_nearest' not in loaded:
                    continue
                base_item = loaded['trishift_nearest']
                truth = np.asarray(base_item['Truth'], dtype=float)
                ctrl = np.asarray(base_item['Ctrl'], dtype=float)
                truth_delta = truth.mean(axis=0) - ctrl.mean(axis=0)
                genes = np.asarray(base_item.get('DE_name', [f'g{i}' for i in range(truth_delta.shape[0])])).astype(str)
                order = np.argsort(-np.abs(truth_delta))[:12]
                rows = [pd.DataFrame({'Gene': genes[order], 'Expression': truth_delta[order], 'Group': 'Truth'})]
                for model_name, item in loaded.items():
                    pred_delta = np.asarray(item['Pred'], dtype=float).mean(axis=0) - np.asarray(item['Ctrl'], dtype=float).mean(axis=0)
                    rows.append(pd.DataFrame({'Gene': genes[order], 'Expression': pred_delta[order], 'Group': MODEL_LABELS[model_name]}))
                case_meta = {'condition': condition_key, 'split_id': split_id, 'subgroup': subgroup, 'score_gain': float(best_case['score_gain'])}
                case_plot_df = pd.concat(rows, ignore_index=True)
                break
            if case_plot_df is not None:
                break
if case_plot_df is not None:
    case_plot_df.to_csv(OUT_ROOT / 'fig4d_combo_case_values.csv', index=False, encoding='utf-8-sig')
    plt.figure(figsize=(14, 5.8), dpi=220)
    ax = sns.barplot(data=case_plot_df, x='Gene', y='Expression', hue='Group', palette=MODEL_COLORS, errorbar=None)
    for patch in ax.patches:
        patch.set_edgecolor('black'); patch.set_linewidth(0.4)
    ax.set_xlabel(''); ax.set_ylabel('Change over control')
    ax.set_title(f"Fig4d: Representative hard-generalization case | {case_meta['subgroup']} | {case_meta['condition']}")
    ax.tick_params(axis='x', rotation=45); ax.legend(title='', frameon=False, ncol=3)
    plt.tight_layout(); plt.savefig(OUT_ROOT / 'fig4d_combo_case.png', bbox_inches='tight'); plt.close()
    print(case_meta); display(case_plot_df.head(20))
else:
    print('No Norman hard-generalization case could be selected.')
print(OUT_ROOT)
